# Autonomous SOC Triage Agent with LlamaIndex & Ollama

This notebook implements an **Autonomous SOC Triage Agent** following the exact architecture of the LlamaIndex Agents & Workflows pattern, powered by a local open-source LLM via **Ollama** (`qwen2.5:14b` by **Alibaba Cloud**).

**Why Ollama + Local LLM for SOC?**
Security logs contain sensitive internal IPs, hostnames, and user identities. Running `qwen2.5:14b` locally via Ollama keeps all telemetry private (zero external LLM API calls) and avoids cloud inference billing errors.

**Contents**
1. Setup & Ollama Configuration
2. Local LLM Integration (`llama-index-llms-ollama`)
3. SOC Playbooks RAG (ChromaDB + `bge-small-en-v1.5`)
4. SOC Triage Tools (`FunctionTool` & `QueryEngineTool`)
5. Autonomous SOC Agent & Multi-Agent System (`AgentWorkflow`)
6. Event-Driven SOC Triage Pipeline (`Workflow`)

## 1. Setup

Install the required LlamaIndex packages and Ollama integration. Make sure Ollama is installed locally (`https://ollama.com`) and pull the model before running:
```bash
ollama pull qwen2.5:14b
```

In [ ]:
%pip install -q "llama-index-core>=0.14.25,<0.15" "llama-index-llms-ollama>=0.5" \
    "llama-index-embeddings-huggingface>=0.8,<0.9" "llama-index-vector-stores-chroma>=0.6,<0.7" \
    chromadb "llama-index-utils-workflow>=0.11,<0.12" python-dotenv

Shared configuration and helpers used throughout the notebook.

In [ ]:
LLM_MODEL = "qwen2.5:14b"  # Provider: Alibaba Cloud (via Ollama)
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
CHROMA_PATH = "./soc_chroma_db"
COLLECTION = "soc_playbooks"


def get_llm(**kwargs):
    """Local Ollama LLM (Alibaba Cloud Qwen 2.5) with native tool-calling support."""
    from llama_index.llms.ollama import Ollama

    return Ollama(
        model=LLM_MODEL,
        request_timeout=120.0,
        temperature=0.1,  # Low temperature for deterministic SOC triage
        **kwargs,
    )


def get_embed_model():
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding

    return HuggingFaceEmbedding(model_name=EMBED_MODEL)

Create a sample SOC Playbook & Runbook knowledge base in `data/soc_playbooks.md` for the RAG pipeline.

In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
Path("data/soc_playbooks.md").write_text(
    """# Enterprise SOC Incident Response Playbooks

## 1. Severity Classification Matrix
- **P1 (Critical)**: Confirmed C2 beaconing, ransomware execution, active data exfiltration, or Domain Admin compromise. SLA: 15 minutes. Auto-containment (`isolate_host`) authorized for confirmed malicious hashes/IPs.
- **P2 (High)**: Brute-force success from external malicious IP, suspicious PowerShell encoded commands, or lateral movement (PsExec/WMI). SLA: 1 hour.
- **P3 (Medium)**: Blocked phishing attempt, repeated failed logins from unknown external IP without success. SLA: 4 hours.
- **P4 (Low / Benign)**: Authorized vulnerability scanner traffic or known false positives.

## 2. Playbook: SSH / RDP Brute Force & Suspicious Login (MITRE T1110)
- Check `check_threat_intel` on source IP.
- Check `query_siem_history` for successful authentication (`EventID 4624` or `Accepted password`) following failures.
- If source IP is malicious AND login succeeded -> Classify **P2 High**, escalate ticket, and recommend/execute host isolation.

## 3. Playbook: Suspicious Outbound C2 / Malware Execution (MITRE T1071 / T1059)
- Parse process name, parent process, destination IP, and file hash.
- If `check_threat_intel` flags destination IP or SHA256 hash as malicious (confidence >= 70) -> Classify **P1 Critical**, call `isolate_host`, and open a P1 incident ticket.
"""
)
print(Path("data/soc_playbooks.md").read_text())

## 2. LlamaHub Integration: Ollama (`qwen2.5:14b`)

Verify that your local Ollama server (`http://localhost:11434`) is responding with `qwen2.5:14b`.

In [ ]:
llm = get_llm()
response = llm.complete("In one sentence, what is the primary goal of Tier-2 SOC triage?")
print(response)

## 3. Components and RAG (SOC Playbooks)

We ingest `data/soc_playbooks.md` into ChromaDB using `IngestionPipeline` and `BAAI/bge-small-en-v1.5`, then wrap it as a `QueryEngine`.

In [ ]:
import chromadb
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore

documents = SimpleDirectoryReader(input_dir="data").load_data()
print(f"Loaded {len(documents)} SOC playbook document(s)")

db = chromadb.PersistentClient(path=CHROMA_PATH)
chroma_collection = db.get_or_create_collection(COLLECTION)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
embed_model = get_embed_model()

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=25),
        embed_model,
    ],
    vector_store=vector_store,
)
stored = await pipeline.arun(documents=documents)
print(f"Stored {len(stored)} node(s) in Chroma")

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
query_engine = index.as_query_engine(llm=llm, similarity_top_k=3, response_mode="compact")
print(query_engine.query("When is isolate_host authorized according to the playbook?"))

## 4. SOC Triage Tools (`FunctionTool` & `QueryEngineTool`)

Define specialized security tools for log parsing, threat intelligence lookup, SIEM correlation, host containment, and ticket escalation.

In [ ]:
import json
import re
from llama_index.core.tools import FunctionTool, QueryEngineTool

# Simulated Threat Intelligence & SIEM stores (replace with VirusTotal/AbuseIPDB/Splunk APIs in production)
MOCK_THREAT_INTEL = {
    "185.220.101.42": {"malicious": True, "confidence": 94, "category": "C2 / Tor Exit Node", "asn": "AS205100"},
    "45.155.205.99": {"malicious": True, "confidence": 88, "category": "SSH Brute Force Scanner", "asn": "AS49870"},
    "8.8.8.8": {"malicious": False, "confidence": 0, "category": "Benign DNS (Google)", "asn": "AS15169"},
    "e99a18c428cb38d5f260853678922e03": {"malicious": True, "confidence": 97, "family": "CobaltStrike Beacon"},
}

MOCK_SIEM_HISTORY = {
    "FIN-WS-042": [
        "2026-11-24T14:10:02Z host=FIN-WS-042 event=powershell_encoded cmd='powershell -enc JABj...' user=a.hassan",
        "2026-11-24T14:11:15Z host=FIN-WS-042 event=outbound_conn dst_ip=185.220.101.42 dst_port=443 bytes=48210",
    ],
    "WEB-SRV-01": [
        "2026-11-24T13:55:00Z host=WEB-SRV-01 event=sshd_failed_login src_ip=45.155.205.99 user=root attempts=42",
        "2026-11-24T13:58:10Z host=WEB-SRV-01 event=sshd_blocked src_ip=45.155.205.99 action=fail2ban_ban",
    ],
}


def parse_security_log(raw_log: str) -> str:
    """Parse a raw security/SIEM log string and extract IPs, hashes, hosts, and users."""
    ips = re.findall(r"\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b", raw_log)
    hashes = re.findall(r"\b[a-fA-F0-9]{32,64}\b", raw_log)
    host_match = re.search(r"host=([A-Za-z0-9_-]+)", raw_log)
    user_match = re.search(r"user=([A-Za-z0-9._-]+)", raw_log)
    result = {
        "ips": list(set(ips)),
        "hashes": list(set(hashes)),
        "host": host_match.group(1) if host_match else "unknown",
        "user": user_match.group(1) if user_match else "unknown",
        "raw_length": len(raw_log),
    }
    return json.dumps(result)


def check_threat_intel(indicator: str) -> str:
    """Check an IP address, domain, or file hash against Threat Intelligence feeds."""
    info = MOCK_THREAT_INTEL.get(
        indicator.strip(),
        {"malicious": False, "confidence": 5, "category": "Unknown / Not in TI feed"},
    )
    return json.dumps({"indicator": indicator, **info})


def query_siem_history(hostname: str) -> str:
    """Retrieve recent SIEM log history for a given hostname to check for lateral movement or prior alerts."""
    events = MOCK_SIEM_HISTORY.get(hostname.strip(), [])
    return json.dumps({"hostname": hostname, "event_count": len(events), "events": events})


def isolate_host(hostname: str, reason: str, dry_run: bool = True) -> str:
    """Isolate a compromised endpoint via EDR/NAC. Defaults to dry_run=True for safety."""
    mode = "DRY-RUN" if dry_run else "EXECUTED"
    return json.dumps({"status": mode, "action": "isolate_host", "hostname": hostname, "reason": reason})


def create_soc_ticket(title: str, severity: str, summary: str, iocs: str) -> str:
    """Create an incident ticket in the SOC case management system (P1-P4)."""
    return json.dumps({
        "ticket_id": "INC-2026-8841",
        "title": title,
        "severity": severity,
        "summary": summary,
        "iocs": iocs,
        "status": "OPEN",
    })


soc_tools = [
    FunctionTool.from_defaults(parse_security_log),
    FunctionTool.from_defaults(check_threat_intel),
    FunctionTool.from_defaults(query_siem_history),
    FunctionTool.from_defaults(isolate_host),
    FunctionTool.from_defaults(create_soc_ticket),
    QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name="soc_playbooks",
        description="Looks up internal SOC incident response playbooks, severity matrix, and MITRE ATT&CK procedures.",
    ),
]
print("Registered SOC tools:", [t.metadata.name for t in soc_tools])

## 5. Autonomous SOC Triage Agent & Multi-Agent System

### 5.1 System Prompt & Single Autonomous Agent with Memory (`Context`)

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.workflow import Context

SOC_SYSTEM_PROMPT = """You are an Autonomous Tier-2 SOC Triage Agent specializing in network security, endpoint telemetry, and log analysis.

Your mission is to triage incoming security alerts, correlate raw logs with threat intelligence and internal SOC playbooks, determine whether an alert is a True Positive, Benign Positive, or False Positive, and recommend or execute containment actions safely.

## Core Triage Workflow
1. **Parse & Extract IOCs**: Always start by calling `parse_security_log` on raw log lines or alert payloads to extract IPs, domains, hashes, usernames, hosts, and event IDs.
2. **Enrich Indicators**: Use `check_threat_intel` on every extracted external IP, domain, or file hash. Never guess reputation from memory.
3. **Correlate Context**: Call `query_siem_history` for the affected host or user to check for lateral movement, brute-force patterns, or prior alerts.
4. **Consult Playbooks**: Query `soc_playbooks` (RAG) to match the observed behavior against official SOC severity criteria, MITRE ATT&CK mappings, and escalation procedures.
5. **Decide & Act**:
   - **False Positive / Benign**: Document evidence and close without containment.
   - **Low / Medium (P3/P4)**: Create a ticket via `create_soc_ticket` with full IOCs and recommended analyst follow-up.
   - **High / Critical (P1/P2) with confirmed malicious IOC or active C2/ransomware**: Call `isolate_host` (with `dry_run=True` unless explicit auto-containment is authorized) and immediately call `create_soc_ticket` with severity P1/P2.

## Guardrails & Output Format
- **Zero Fabrication**: Never invent IP addresses, hashes, timestamps, or CVE/MITRE IDs not present in tool outputs.
- **Safety First**: Treat `isolate_host` as a high-impact action; only trigger it when threat intel or SIEM correlation confirms malicious activity.
- **Structured Final Report**: Conclude every investigation with:
  - **Verdict**: (True Positive / False Positive / Suspicious - Needs Review)
  - **Severity**: (P1 Critical / P2 High / P3 Medium / P4 Low)
  - **MITRE ATT&CK Mapping**: (Tactic & Technique ID if applicable)
  - **Key Evidence & IOCs**: Bullet list from tool outputs
  - **Actions Taken / Recommended Next Steps**"""

soc_agent = AgentWorkflow.from_tools_or_functions(
    soc_tools,
    llm=llm,
    system_prompt=SOC_SYSTEM_PROMPT,
)

ctx = Context(soc_agent)

sample_alert = (
    "ALERT: Suspicious PowerShell & Outbound Connection | "
    "2026-11-24T14:11:20Z host=FIN-WS-042 user=a.hassan "
    "src_ip=10.10.20.15 dst_ip=185.220.101.42 "
    "proc_hash=e99a18c428cb38d5f260853678922e03"
)

result = await soc_agent.run(user_msg=f"Triage this alert:\n{sample_alert}", ctx=ctx)
print(result)

### 5.2 Follow-up Investigation in the Same Context

Because `ctx` preserves conversation state, the analyst can ask follow-up questions about the same incident.

In [ ]:
follow_up = await soc_agent.run(
    user_msg="Should we move FIN-WS-042 out of dry_run isolation? Explain based on the playbook.",
    ctx=ctx,
)
print(follow_up)

### 5.3 Multi-Agent SOC Architecture (`Triage Analyst` + `Containment Responder`)

In [ ]:
from llama_index.core.agent.workflow import ReActAgent

triage_analyst = ReActAgent(
    name="triage_analyst",
    description="Parses security logs, enriches IOCs with Threat Intel, checks SIEM history, and consults SOC playbooks.",
    system_prompt=(
        "You are a Tier-2 SOC Triage Analyst. Parse logs, check threat intel, correlate SIEM history, "
        "and check soc_playbooks. Hand off to containment_responder when a P1/P2 incident requires ticket creation or host isolation."
    ),
    tools=soc_tools[:3] + [soc_tools[5]],  # parse, threat_intel, siem_history, soc_playbooks
    llm=llm,
)

containment_responder = ReActAgent(
    name="containment_responder",
    description="Executes host isolation and creates SOC incident tickets for confirmed threats.",
    system_prompt=(
        "You are a SOC Incident Responder. Create structured SOC tickets and execute host isolation "
        "(dry_run=True by default unless explicitly authorized) based on verified triage findings."
    ),
    tools=[soc_tools[3], soc_tools[4]],  # isolate_host, create_soc_ticket
    llm=llm,
)

soc_multi_agent = AgentWorkflow(
    agents=[triage_analyst, containment_responder],
    root_agent="triage_analyst",
)

print(await soc_multi_agent.run(user_msg=f"Investigate and respond to this alert:\n{sample_alert}"))

## 6. Event-Driven SOC Triage Workflow

For deterministic production pipelines, `Workflow` enforces step-by-step execution with typed `Event`s and shared `Context` state.

In [ ]:
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step
from llama_index.utils.workflow import draw_all_possible_flows


class ParsedLogEvent(Event):
    parsed_data: dict


class EnrichedAlertEvent(Event):
    parsed_data: dict
    ti_findings: list
    is_critical: bool


class SOCTriageWorkflow(Workflow):
    @step
    async def ingest_and_parse(self, ctx: Context, ev: StartEvent) -> ParsedLogEvent:
        raw_log = ev.get("raw_log", "")
        parsed = json.loads(parse_security_log(raw_log))
        await ctx.store.set("host", parsed["host"])
        return ParsedLogEvent(parsed_data=parsed)

    @step
    async def enrich_iocs(self, ctx: Context, ev: ParsedLogEvent) -> EnrichedAlertEvent:
        indicators = ev.parsed_data["ips"] + ev.parsed_data["hashes"]
        findings = [json.loads(check_threat_intel(ind)) for ind in indicators]
        is_critical = any(f.get("malicious") and f.get("confidence", 0) >= 70 for f in findings)
        return EnrichedAlertEvent(
            parsed_data=ev.parsed_data,
            ti_findings=findings,
            is_critical=is_critical,
        )

    @step
    async def decide_and_escalate(self, ctx: Context, ev: EnrichedAlertEvent) -> StopEvent:
        host = await ctx.store.get("host")
        if ev.is_critical:
            isolation = json.loads(isolate_host(host, "Confirmed high-confidence malicious IOC", dry_run=True))
            ticket = json.loads(
                create_soc_ticket(
                    title=f"P1 Critical Alert on {host}",
                    severity="P1",
                    summary="Automated workflow detected high-confidence malicious IOCs.",
                    iocs=json.dumps(ev.ti_findings),
                )
            )
            return StopEvent(result={"verdict": "TRUE_POSITIVE", "isolation": isolation, "ticket": ticket})
        return StopEvent(result={"verdict": "LOW_RISK_OR_BENIGN", "host": host, "ti_findings": ev.ti_findings})


soc_wf = SOCTriageWorkflow(timeout=30, verbose=True)
draw_all_possible_flows(SOCTriageWorkflow, "soc_triage_flow.html")

wf_result = await soc_wf.run(
    raw_log="host=FIN-WS-042 user=a.hassan dst_ip=185.220.101.42 hash=e99a18c428cb38d5f260853678922e03"
)
print(json.dumps(wf_result, indent=2))